# Model matrix validation against hand calculations

This notebook runs `solve_network()` with the committed network data, displays the assembled linear system, and compares it with the hand-calculation reference. It intentionally refuses to silently truncate malformed reference diagonals.

In [ ]:
import numpy as np
from model import NODE_TREE, BRANCHES, solve_network
from tests.test_model import (
    HAND_UNKNOWN_ORDER, HAND_A_DIAGONALS, HAND_B,
    align_solver_matrix, build_handcalc_matrix,
)

## 1. Run the solver and inspect the system

The solver now returns `matrix`, `rhs`, and `unknowns` in its native ordering. We reorder the system to match the handwritten unknown order before comparing.

In [ ]:
results = solve_network(NODE_TREE, BRANCHES)
A_solver, b_solver = align_solver_matrix(results)

print('Solver unknown order:', results['unknowns'])
print('Hand-calculation order:', HAND_UNKNOWN_ORDER)
print('A from code, aligned to hand-calculation order:')
display(A_solver)
print('b from code, aligned to hand-calculation order:')
display(b_solver)

## 2. Validate the hand-calculation reference data

The committed `A_diagonals` contain more entries than a 7×7 matrix can fit on offsets `+1` and `-1`. This cell reports that issue rather than silently discarding values. Correct those reference entries before treating any matrix comparison as valid.

In [ ]:
try:
    A_hand = build_handcalc_matrix()
except ValueError as exc:
    A_hand = None
    print('REFERENCE DATA ERROR:', exc)
    print('\nDiagonal lengths supplied vs required:')
    n = len(HAND_UNKNOWN_ORDER)
    for offset, values in HAND_A_DIAGONALS.items():
        print(f'offset {offset:+}: supplied {len(values)}, required {n - abs(offset)}')
else:
    print('Hand-calculated A:')
    display(A_hand)
    print('Hand-calculated b:')
    display(HAND_B)

## 3. Compare A and b (runs once the reference matrix is valid)

`A` and `b` are compared independently. A passing result means numerical agreement within the stated tolerances; it does not replace the separate mass-conservation test.

In [ ]:
if A_hand is None:
    print('Comparison blocked: correct the malformed hand-calculation diagonals above, then rerun this cell.')
else:
    a_match = np.allclose(A_solver, A_hand, rtol=1e-6, atol=1e-9)
    b_match = np.allclose(b_solver, HAND_B, rtol=1e-6, atol=1e-6)
    print('A matches:', a_match)
    print('b matches:', b_match)
    print('A difference (code - hand):')
    display(A_solver - A_hand)
    print('b difference (code - hand):')
    display(b_solver - HAND_B)
    assert a_match and b_match, 'Solver system differs from the hand-calculated system'

## 4. Run the automated tests

From the repository root, run `pytest -q`. The reference-diagonal validation test is expected to fail until the recorded hand-calculation diagonal lengths are corrected; this is deliberate and prevents a false-positive validation.

In [ ]:
# In a terminal from the repository root, run: pytest -q
# Or run the test suite in this notebook environment:
import pytest
pytest.main(['-q', 'tests/test_model.py'])